# Proyecto RappiPlus: de datos a decisiones de negocio

**Introducción**


El objetivo de este proyecto es evaluar el desempeño del servicio **RappiPlus** para apoyar **decisiones de negocio basadas en datos**.

Se trabajan con múltiples datasets del negocio:

- **rappiplus_orders_raw.csv** → información de pedidos, precios, descuentos y revenue  
- **rappiplus_catalog.csv** → costos de productos, categorías y proveedores  
- **rappiplus_marketing_spend.csv** → inversión en marketing por canal y país  
- **events / users / user_activity (SQL)** → comportamiento del usuario dentro de la plataforma  
- **experiment_checkout_ui.csv** → resultados de un experimento A/B en el checkout  

El análisis sigue una lógica clara y progresiva:

1. 🔍 Evaluar si podemos confiar en los datos (calidad de datos en Python) 

2. 💰 Analizar si el negocio es rentable (revenue, costos y profit)  

3. 🛒 Entender dónde se pierden los usuarios (funnel de conversión)  

4. 🔁 Evaluar si los usuarios regresan (retención por cohortes)  

5. 🧪 Validar si los cambios generan impacto (test estadístico)  

6. 📊 Comunicar los resultados (dashboard en BI)  

A lo largo del proyecto, se transforman datos en insights para responder preguntas clave del negocio y proponer **recomendaciones accionables**.

---

## 🔹 Paso 1: Cargar y validar la calidad de los datos

---

### 1.1 Carga de datos y vista rápida

**🎯 Objetivo:** Familiarizarte con la estructura de los datasets del negocio antes de analizarlos.

**Instrucciones:**

- Importa las librerías necesarias
- Carga los archivos:
  - `rappiplus_orders_raw.csv`
  - `rappiplus_catalog.csv`
  - `rappiplus_marketing_spend.csv`
- Guarda los DataFrames en:
  - `orders`, `catalog`, `marketing`
- Explora cada dataset.

---

In [1]:
# importar librerías

# importar librerías
import pandas as pd
import numpy as np



In [2]:

# cargar archivos
orders = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_orders_raw.csv')
catalog = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_catalog.csv')
marketing = pd.read_csv('https://practicum-content.s3.amazonaws.com/datasets/rappiplus_marketing_spend.csv')





In [3]:

# explorar datasets
print("Dimensiones de orders:", orders.shape)
display(orders.head())

print("Dimensiones de catalog:", catalog.shape)
display(catalog.head())

print("Dimensiones de marketing:", marketing.shape)
display(marketing.head())


# revisar estructura y tipos de datos
print("=== ORDERS ===")
orders.info()
print("\nValores nulos:")
print(orders.isnull().sum())

print("\n=== CATALOG ===")
catalog.info()
print("\nValores nulos:")
print(catalog.isnull().sum())

print("\n=== MARKETING ===")
marketing.info()
print("\nValores nulos:")
print(marketing.isnull().sum())


# revisar duplicados
print("Duplicados completos en orders:", orders.duplicated().sum())
print("IDs de pedido duplicados:", orders['id_pedido'].duplicated().sum())
print("Duplicados completos en catalog:", catalog.duplicated().sum())
print("Duplicados completos en marketing:", marketing.duplicated().sum())

# revisar valores únicos de las columnas categóricas
print("\nPaíses en orders:")
print(orders['pais'].value_counts(dropna=False))

print("\nCategorías en orders:")
print(orders['categoria_producto'].value_counts(dropna=False))

print("\nCategorías en catalog:")
print(catalog['categoria_producto'].value_counts(dropna=False))

print("\nCanales en marketing:")
print(marketing['canal'].value_counts(dropna=False))


# inspeccionar pedidos con IDs repetidos
pedidos_repetidos = orders[
    orders.duplicated(subset='id_pedido', keep=False)
].sort_values('id_pedido')

print("Filas con IDs de pedido repetidos:", len(pedidos_repetidos))
print("Cantidad de IDs de pedido afectados:",
      pedidos_repetidos['id_pedido'].nunique())

# comprobar si los registros de un mismo ID tienen datos diferentes
variacion_por_id = orders.groupby('id_pedido').nunique(dropna=False)
ids_con_diferencias = variacion_por_id[
    variacion_por_id.gt(1).any(axis=1)
].index

print("IDs repetidos con datos diferentes:", len(ids_con_diferencias))

display(pedidos_repetidos.head(20))


# confirmar duplicados exactos y conservar el original
orders_original = orders.copy()

print("Filas originales:", len(orders_original))
print("Duplicados exactos:", orders_original.duplicated().sum())
print("IDs de pedido únicos:", orders_original['id_pedido'].nunique())

# conservar una copia de los datos originales
orders_original = orders.copy()

# eliminar únicamente las filas completamente duplicadas
orders = orders.drop_duplicates().copy()

# verificar los resultados de la limpieza
print("Filas originales:", len(orders_original))
print("Filas después de eliminar duplicados:", len(orders))
print("Duplicados exactos restantes:", orders.duplicated().sum())
print("IDs de pedido únicos:", orders['id_pedido'].nunique())

# revisar los tipos de datos después de eliminar duplicados
print("=== ORDERS ===")
print(orders.dtypes)

print("\n=== CATALOG ===")
print(catalog.dtypes)

print("\n=== MARKETING ===")
print(marketing.dtypes)


# revisar el formato de las fechas
print("Fechas de pedidos:")
display(orders['fecha_hora_pedido'].head(10))

print("\nFechas de marketing:")
display(marketing['fecha'].head(10))


# convertir las columnas de fechas
orders['fecha_hora_pedido'] = pd.to_datetime(orders['fecha_hora_pedido'])
marketing['fecha'] = pd.to_datetime(marketing['fecha'])

# verificar las conversiones
print("Tipo de fecha en orders:", orders['fecha_hora_pedido'].dtype)
print("Tipo de fecha en marketing:", marketing['fecha'].dtype)

print("\nRango de fechas de pedidos:")
print(orders['fecha_hora_pedido'].min(), "a",
      orders['fecha_hora_pedido'].max())

print("\nRango de fechas de marketing:")
print(marketing['fecha'].min(), "a",
      marketing['fecha'].max())

# revisar los valores nulos después de eliminar duplicados
print("=== ORDERS ===")
print(orders.isnull().sum())

print("\n=== CATALOG ===")
print(catalog.isnull().sum())

print("\n=== MARKETING ===")
print(marketing.isnull().sum())

print("=== ORDERS ===")
print(orders.isnull().sum())

# revisar filas con valores nulos en orders
filas_con_nulos = orders[orders.isnull().any(axis=1)]

print("Filas con al menos un valor nulo:", len(filas_con_nulos))

print("\nCantidad de nulos por fila:")
display(filas_con_nulos.isnull().sum(axis=1).value_counts().sort_index())

print("\nPrimeras 10 filas con valores nulos:")
display(filas_con_nulos.head(10))

# contar filas afectadas por valores nulos
filas_con_nulos = orders[orders.isnull().any(axis=1)]

print("Filas con al menos un valor nulo:", len(filas_con_nulos))

print("\nCantidad de nulos por fila:")
display(
    filas_con_nulos.isnull().sum(axis=1).value_counts().sort_index()
)

print("\nFilas con nulos por columna:")
print(filas_con_nulos.isnull().sum())

# identificar las combinaciones de valores nulos
columnas_nulas = orders.columns[orders.isnull().any()]

patrones_nulos = (
    orders[columnas_nulas]
    .isnull()
    .astype(int)
    .value_counts()
    .reset_index(name='cantidad_filas')
)

display(patrones_nulos)

# revisar pedidos con datos numéricos faltantes
pedidos_numericos_incompletos = orders[
    orders[['cantidad', 'precio_unitario', 'monto_descuento']]
    .isnull()
    .any(axis=1)
]

display(
    pedidos_numericos_incompletos[
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ]
)

# revisar la relación entre cantidad, precio, descuento y total
pedidos_completos = orders.dropna(
    subset=['cantidad', 'precio_unitario', 'monto_descuento']
).copy()

pedidos_completos['total_calculado'] = (
    pedidos_completos['cantidad'] * pedidos_completos['precio_unitario']
    - pedidos_completos['monto_descuento']
)

pedidos_completos['diferencia_total'] = (
    pedidos_completos['monto_total']
    - pedidos_completos['total_calculado']
)

print("Pedidos evaluados:", len(pedidos_completos))
print(
    "Pedidos donde el total coincide, con tolerancia de 0.01:",
    np.isclose(
        pedidos_completos['monto_total'],
        pedidos_completos['total_calculado'],
        atol=0.01
    ).sum()
)
print(
    "Diferencia absoluta máxima:",
    pedidos_completos['diferencia_total'].abs().max()
)

display(
    pedidos_completos[
        [
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total',
            'total_calculado',
            'diferencia_total'
        ]
    ].head(10)
)

# identificar pedidos cuyo total no coincide con la fórmula
pedidos_con_diferencias = pedidos_completos[
    ~np.isclose(
        pedidos_completos['monto_total'],
        pedidos_completos['total_calculado'],
        atol=0.01
    )
]

print("Pedidos con diferencias:", len(pedidos_con_diferencias))

display(
    pedidos_con_diferencias[
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total',
            'total_calculado',
            'diferencia_total'
        ]
    ]
)

# revisar pedidos con cantidades negativas
pedidos_cantidad_negativa = orders[
    orders['cantidad'] < 0
]

print(
    "Pedidos con cantidad negativa:",
    len(pedidos_cantidad_negativa)
)

display(
    pedidos_cantidad_negativa[
        [
            'id_pedido',
            'fecha_hora_pedido',
            'pais',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ]
)

# revisar posibles anomalías en columnas numéricas
print("Cantidades iguales a cero:", (orders['cantidad'] == 0).sum())
print("Precios negativos:", (orders['precio_unitario'] < 0).sum())
print("Descuentos negativos:", (orders['monto_descuento'] < 0).sum())

print("\nPedidos con cantidad cero:")
display(orders[orders['cantidad'] == 0])

print("\nPedidos con precio negativo:")
display(orders[orders['precio_unitario'] < 0])

print("\nPedidos con descuento negativo:")
display(orders[orders['monto_descuento'] < 0])

# revisar los valores únicos de las columnas categóricas
columnas_categoricas = [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'categoria_producto'
]

for columna in columnas_categoricas:
    print(f"\n=== {columna.upper()} ===")
    print(orders[columna].value_counts(dropna=False))

    # normalizar los nombres de los países sin modificar los nulos
orders['pais'] = orders['pais'].str.strip().str.title()

# verificar los resultados
print(orders['pais'].value_counts(dropna=False))

print("Categorías en orders:")
print(orders['categoria_producto'].value_counts(dropna=False))

print("\nCategorías en catalog:")
print(catalog['categoria_producto'].value_counts(dropna=False))

print("\nProductos en orders:")
print(sorted(orders['nombre_producto'].dropna().unique()))

print("\nProductos en catalog:")
print(sorted(catalog['nombre_producto'].unique()))

# unificar la categoría de electrónica
orders['categoria_producto'] = orders['categoria_producto'].replace(
    {'Electronica': 'Electrónica'}
)

# verificar las categorías normalizadas
print("Categorías en orders:")
print(orders['categoria_producto'].value_counts(dropna=False))

print("\nCategorías en catalog:")
print(catalog['categoria_producto'].value_counts(dropna=False))

# revisar pedidos con categoría faltante
pedidos_sin_categoria = orders[
    orders['categoria_producto'].isnull()
]

print("Pedidos sin categoría:", len(pedidos_sin_categoria))
print(
    "Con nombre de producto disponible:",
    pedidos_sin_categoria['nombre_producto'].notna().sum()
)
print(
    "Sin nombre de producto:",
    pedidos_sin_categoria['nombre_producto'].isna().sum()
)

display(
    pedidos_sin_categoria[
        ['id_pedido', 'nombre_producto', 'categoria_producto']
    ].head(15)
)

# crear un diccionario de productos y categorías del catálogo
mapa_categorias = catalog.set_index(
    'nombre_producto'
)['categoria_producto']

# identificar las categorías que faltan y tienen producto conocido
mascara_categoria_faltante = (
    orders['categoria_producto'].isnull()
    & orders['nombre_producto'].notna()
)

# recuperar las categorías utilizando el catálogo
orders.loc[
    mascara_categoria_faltante,
    'categoria_producto'
] = orders.loc[
    mascara_categoria_faltante,
    'nombre_producto'
].map(mapa_categorias)

# verificar el resultado
print("Categorías nulas restantes:", orders['categoria_producto'].isnull().sum())

print("\nDistribución de categorías:")
print(orders['categoria_producto'].value_counts(dropna=False))

# revisar pedidos sin nombre de producto
pedidos_sin_producto = orders[
    orders['nombre_producto'].isnull()
]

print("Pedidos sin nombre de producto:", len(pedidos_sin_producto))

print("\nValores nulos por columna en esos pedidos:")
print(pedidos_sin_producto.isnull().sum())

print("\nMuestra de los pedidos:")
display(pedidos_sin_producto.head(10))

# comprobar el estado actual de los datasets
print("ORDERS:", orders.shape)
print("Duplicados exactos:", orders.duplicated().sum())
print("Tipo de fecha en orders:", orders['fecha_hora_pedido'].dtype)

print("\nPaíses:")
print(orders['pais'].value_counts(dropna=False))

print("\nCategorías:")
print(orders['categoria_producto'].value_counts(dropna=False))

Dimensiones de orders: (25100, 12)


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99
3,order_3,user_4510,2025-06-09,Colombia,mobile,social,Tablet-Standard-64GB,Electronica,1.0,257.87,15.0,242.87
4,order_4,user_5044,2025-03-30,Argentina,desktop,paid_search,Blender-XL-Red,Hogar,1.0,336.28,0.0,336.28


Dimensiones de catalog: (7, 4)


,nombre_producto,categoria_producto,costo_unitario,proveedor
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
3,Blender-XL-Red,Hogar,176.64,Long-Reid
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


Dimensiones de marketing: (1620, 5)


...


=== PAIS ===
Colombia     7481
Mexico       7478
Argentina    7259
mexico        863
colombia      823
argentina     796
NaN           300
Name: pais, dtype: int64

=== DISPOSITIVO ===
desktop    12711
mobile     12269
NaN           20
Name: dispositivo, dtype: int64

=== FUENTE_REFERENCIA ===
social         8402
organic        8288
paid_search    8280
NaN              30
Name: fuente_referencia, dtype: int64

=== CATEGORIA_PRODUCTO ===
Hogar          8346
Moda           8295
Electronica    8279
NaN              80
Name: categoria_producto, dtype: int64
Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: pais, dtype: int64
Categorías en orders:
Hogar          8346
Moda           8295
Electronica    8279
NaN              80
Name: categoria_producto, dtype: int64

Categorías en catalog:
Electrónica    3
Moda           2
Hogar          2
Name: categoria_producto, dtype: int64

Productos en orders:
['Blender-XL-Red', 'Jacket-Winter-M', 'Laptop-Gaming-16GB', 'Phon

,id_pedido,nombre_producto,categoria_producto
44,order_44,NaN,NaN
45,order_45,NaN,NaN
46,order_46,NaN,NaN
47,order_47,NaN,NaN
48,order_48,NaN,NaN
49,order_49,NaN,NaN
50,order_50,NaN,NaN
51,order_51,NaN,NaN
52,order_52,NaN,NaN
53,order_53,NaN,NaN


Categorías nulas restantes: 30

Distribución de categorías:
Hogar          8355
Moda           8324
Electrónica    8291
NaN              30
Name: categoria_producto, dtype: int64
Pedidos sin nombre de producto: 30

Valores nulos por columna en esos pedidos:
id_pedido              0
id_usuario             0
fecha_hora_pedido      0
pais                   0
dispositivo            0
fuente_referencia     30
nombre_producto       30
categoria_producto    30
cantidad               0
precio_unitario        0
monto_descuento        0
monto_total            0
dtype: int64

Muestra de los pedidos:


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
44,order_44,user_2899,2025-02-22,Colombia,desktop,NaN,NaN,NaN,1.0,318.38,5.0,313.38
45,order_45,user_6196,2025-02-10,Mexico,mobile,NaN,NaN,NaN,2.0,354.06,10.0,698.12
46,order_46,user_5815,2025-04-24,Mexico,mobile,NaN,NaN,NaN,1.0,359.31,10.0,349.31
47,order_47,user_406,2025-05-08,Colombia,mobile,NaN,NaN,NaN,2.0,476.78,0.0,953.56
48,order_48,user_6187,2025-01-26,Colombia,desktop,NaN,NaN,NaN,2.0,137.96,0.0,275.93
49,order_49,user_7225,2025-04-04,Argentina,desktop,NaN,NaN,NaN,2.0,433.76,0.0,867.53
50,order_50,user_1840,2025-02-05,Mexico,desktop,NaN,NaN,NaN,1.0,495.38,5.0,490.38
51,order_51,user_5998,2025-02-21,Mexico,desktop,NaN,NaN,NaN,2.0,382.94,0.0,765.89
52,order_52,user_5679,2025-04-17,Argentina,mobile,NaN,NaN,NaN,2.0,415.01,0.0,830.01
53,order_53,user_5936,2025-01-02,Argentina,desktop,NaN,NaN,NaN,2.0,236.49,0.0,472.98


ORDERS: (25000, 12)
Duplicados exactos: 0
Tipo de fecha en orders: datetime64[ns]

Países:
Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: pais, dtype: int64

Categorías:
Hogar          8355
Moda           8324
Electrónica    8291
NaN              30
Name: categoria_producto, dtype: int64


---

### Revisión y calidad de datos

**🎯 Objetivo:** Detectar y corregir problemas en los datos que puedan afectar el análisis de revenue, costos y rentabilidad.

Se revisan los 3 datasets
- Validar y convertir fechas al formato correcto  
- Revisar variables numéricas (sin negativos o ceros inválidos)  
- Verificar consistencia de montos  
- Eliminar duplicados  
- Revisar variables categóricas 

---

In [4]:
# 1.2.1. Validación de calidad de orders

# Revisar dimensiones y duplicados
print("Dimensiones de orders:", orders.shape)
print("Duplicados exactos:", orders.duplicated().sum())
print("IDs de pedido duplicados:", orders['id_pedido'].duplicated().sum())

# Revisar fechas
print("\nTipo de fecha:", orders['fecha_hora_pedido'].dtype)
print(
    "Rango de fechas:",
    orders['fecha_hora_pedido'].min(),
    "a",
    orders['fecha_hora_pedido'].max()
)

# Revisar valores nulos
print("\nValores nulos por columna:")
print(orders.isnull().sum())

# Revisar cantidades, precios y descuentos
print("\nCantidades negativas:", (orders['cantidad'] < 0).sum())
print("Cantidades iguales a cero:", (orders['cantidad'] == 0).sum())
print("Precios negativos:", (orders['precio_unitario'] < 0).sum())
print("Descuentos negativos:", (orders['monto_descuento'] < 0).sum())

Dimensiones de orders: (25000, 12)
Duplicados exactos: 0
IDs de pedido duplicados: 0

Tipo de fecha: datetime64[ns]
Rango de fechas: 2025-01-01 00:00:00 a 2025-06-30 00:00:00

Valores nulos por columna:
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

Cantidades negativas: 4
Cantidades iguales a cero: 0
Precios negativos: 0
Descuentos negativos: 0


In [5]:
# 1.2.2. Validar la consistencia de los importes

# Usar solo pedidos con los tres datos numéricos disponibles
pedidos_validos = orders.dropna(
    subset=['cantidad', 'precio_unitario', 'monto_descuento']
).copy()

# Calcular el total esperado
pedidos_validos['total_calculado'] = (
    pedidos_validos['cantidad'] * pedidos_validos['precio_unitario']
    - pedidos_validos['monto_descuento']
)

# Calcular la diferencia frente al total registrado
pedidos_validos['diferencia_total'] = (
    pedidos_validos['monto_total']
    - pedidos_validos['total_calculado']
)

# Resumir los resultados
coinciden = np.isclose(
    pedidos_validos['monto_total'],
    pedidos_validos['total_calculado'],
    atol=0.01
)

print("Pedidos evaluados:", len(pedidos_validos))
print("Pedidos que coinciden:", coinciden.sum())
print("Pedidos con diferencias:", (~coinciden).sum())

print(
    "Diferencia absoluta máxima:",
    pedidos_validos['diferencia_total'].abs().max()
)

# Mostrar los pedidos con diferencias
display(
    pedidos_validos.loc[
        ~coinciden,
        [
            'id_pedido',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total',
            'total_calculado',
            'diferencia_total'
        ]
    ]
)

Pedidos evaluados: 24950
Pedidos que coinciden: 24947
Pedidos con diferencias: 3
Diferencia absoluta máxima: 20.0


,id_pedido,cantidad,precio_unitario,monto_descuento,monto_total,total_calculado,diferencia_total
266,order_266,-2.0,101.31,10.0,-192.62,-212.62,20.0
267,order_267,-1.0,43.50,5.0,-38.50,-48.50,10.0
268,order_268,-1.0,497.65,5.0,-492.65,-502.65,10.0


In [6]:
# 1.2.3. Validar el catálogo de productos

print("Dimensiones de catalog:", catalog.shape)
print("Duplicados exactos:", catalog.duplicated().sum())
print(
    "Productos duplicados:",
    catalog['nombre_producto'].duplicated().sum()
)

print("\nValores nulos por columna:")
print(catalog.isnull().sum())

print("\nCostos iguales o menores que cero:")
print((catalog['costo_unitario'] <= 0).sum())

print("\nDetalle de productos y costos:")
display(
    catalog[
        [
            'nombre_producto',
            'categoria_producto',
            'costo_unitario',
            'proveedor'
        ]
    ].sort_values('nombre_producto')
)

Dimensiones de catalog: (7, 4)
Duplicados exactos: 0
Productos duplicados: 0

Valores nulos por columna:
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

Costos iguales o menores que cero:
0

Detalle de productos y costos:


,nombre_producto,categoria_producto,costo_unitario,proveedor
3,Blender-XL-Red,Hogar,176.64,Long-Reid
6,Jacket-Winter-M,Moda,189.31,Mcmillan-Rhodes
0,Laptop-Gaming-16GB,Electrónica,280.68,"Fuller, Pena and Myers"
1,Phone-Pro-128GB,Electrónica,10.12,King Ltd
5,Sneakers-Urban-42,Moda,17.21,Greene-Smith
2,Tablet-Standard-64GB,Electrónica,25.21,Bowers LLC
4,Vacuum-Pro-Black,Hogar,16.60,"Rivera, Carr and Finley"


In [7]:
# 1.2.4. Validar el dataset de marketing

print("Dimensiones de marketing:", marketing.shape)
print("Duplicados exactos:", marketing.duplicated().sum())

print("\nTipos de datos:")
print(marketing.dtypes)

print("\nValores nulos por columna:")
print(marketing.isnull().sum())

print("\nGastos negativos:", (marketing['gasto'] < 0).sum())
print("Gastos iguales a cero:", (marketing['gasto'] == 0).sum())

print("\nRango de fechas:")
print(marketing['fecha'].min(), "a", marketing['fecha'].max())

print("\nCanales registrados:")
print(marketing['canal'].value_counts(dropna=False))

Dimensiones de marketing: (1620, 5)
Duplicados exactos: 0

Tipos de datos:
fecha         datetime64[ns]
pais                  object
id_campaña            object
canal                 object
gasto                float64
dtype: object

Valores nulos por columna:
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64

Gastos negativos: 0
Gastos iguales a cero: 0

Rango de fechas:
2025-01-01 00:00:00 a 2025-06-29 00:00:00

Canales registrados:
paid_search    507
organic        506
social         506
NaN            101
Name: canal, dtype: int64


In [8]:
# 1.2.5. Revisar consistencia de países en marketing
print("Países en marketing:")
print(marketing['pais'].value_counts(dropna=False))

print("\nPaíses únicos en marketing:")
print(sorted(marketing['pais'].dropna().unique()))

Países en marketing:
Mexico       540
Argentina    540
Colombia     540
Name: pais, dtype: int64

Países únicos en marketing:
['Argentina', 'Colombia', 'Mexico']


In [9]:
# 1.2.6. Investigar los canales faltantes en marketing

marketing_sin_canal = marketing[marketing['canal'].isnull()]

print("Registros sin canal:", len(marketing_sin_canal))

print("\nRegistros sin canal por país:")
print(marketing_sin_canal['pais'].value_counts())

print("\nRegistros sin canal por campaña:")
print(marketing_sin_canal['id_campaña'].value_counts())

print("\nFechas de los registros sin canal:")
print(
    marketing_sin_canal['fecha'].min(),
    "a",
    marketing_sin_canal['fecha'].max()
)

Registros sin canal: 101

Registros sin canal por país:
Argentina    34
Mexico       34
Colombia     33
Name: pais, dtype: int64

Registros sin canal por campaña:
organic_Mexico           12
social_Argentina         12
organic_Colombia         11
social_Colombia          11
paid_search_Argentina    11
organic_Argentina        11
social_Mexico            11
paid_search_Mexico       11
paid_search_Colombia     11
Name: id_campaña, dtype: int64

Fechas de los registros sin canal:
2025-01-11 00:00:00 a 2025-01-23 00:00:00


In [10]:
# 1.2.7. Verificación final antes de exportar

for nombre, df in [
    ('orders', orders),
    ('catalog', catalog),
    ('marketing', marketing)
]:
    print(f"\n=== {nombre.upper()} ===")
    print("Dimensiones:", df.shape)
    print("Duplicados exactos:", df.duplicated().sum())
    print("Total de valores nulos:", df.isnull().sum().sum())
    print("\nNulos por columna:")
    print(df.isnull().sum())


=== ORDERS ===
Dimensiones: (25000, 12)
Duplicados exactos: 0
Total de valores nulos: 560

Nulos por columna:
id_pedido               0
id_usuario              0
fecha_hora_pedido       0
pais                  300
dispositivo            20
fuente_referencia      30
nombre_producto        30
categoria_producto     30
cantidad               50
precio_unitario        50
monto_descuento        50
monto_total             0
dtype: int64

=== CATALOG ===
Dimensiones: (7, 4)
Duplicados exactos: 0
Total de valores nulos: 0

Nulos por columna:
nombre_producto       0
categoria_producto    0
costo_unitario        0
proveedor             0
dtype: int64

=== MARKETING ===
Dimensiones: (1620, 5)
Duplicados exactos: 0
Total de valores nulos: 101

Nulos por columna:
fecha           0
pais            0
id_campaña      0
canal         101
gasto           0
dtype: int64


In [11]:
# revisar categorías y valores de texto únicos
for columna in [
    'pais',
    'dispositivo',
    'fuente_referencia',
    'categoria_producto'
]:
    print(f"\n=== {columna.upper()} ===")
    print(orders[columna].value_counts(dropna=False))


=== PAIS ===
Mexico       8341
Colombia     8304
Argentina    8055
NaN           300
Name: pais, dtype: int64

=== DISPOSITIVO ===
desktop    12711
mobile     12269
NaN           20
Name: dispositivo, dtype: int64

=== FUENTE_REFERENCIA ===
social         8402
organic        8288
paid_search    8280
NaN              30
Name: fuente_referencia, dtype: int64

=== CATEGORIA_PRODUCTO ===
Hogar          8355
Moda           8324
Electrónica    8291
NaN              30
Name: categoria_producto, dtype: int64


---
**📦 Exportación**: Una vez finalizada la limpieza, se exportan los datasets para utilizarlos en la última etapa del proyecto.

In [12]:
# exportar datasets limpios
orders.to_csv('orders_clean.csv', index=False)
catalog.to_csv('catalog_clean.csv', index=False)
marketing.to_csv('marketing_clean.csv', index=False)

In [13]:
import os

print("Carpeta actual:", os.getcwd())

print("\nArchivos CSV generados:")
for archivo in ["orders_clean.csv", "catalog_clean.csv", "marketing_clean.csv"]:
    print(archivo, "->", os.path.exists(archivo))

Carpeta actual: /app

Archivos CSV generados:
orders_clean.csv -> True
catalog_clean.csv -> True
marketing_clean.csv -> True


---

## 🔹 Paso 2: Analizar si el negocio es rentable

### 2.1 Cálculo de KPIs principales

**🎯 Objetivo:** Calcular los indicadores clave del negocio para evaluar ingresos, costos y rentabilidad.

Se usan los 3 datasets (`orders`, `catalog`, `marketing_spend`):

**📊 Parte 1: Rentabilidad del negocio**
- ¿Cuál es el ingreso total (revenue)? 
- ¿Cuál es el costo total? 
- ¿Cuánto se ha invertido en marketing? 
- ¿El negocio es rentable? (calcular profit)  

---

**📈 Parte 2: Comportamiento de ventas**
- ¿Cuál es el ticket promedio por orden? 
- ¿Cuál es la cantidad promedio de productos por orden? 
- ¿Cuál es el producto más vendido?
- ¿Cuánto se ha gastado en marketing por canal? 

In [14]:
# tu código aquí

# Ingreso total del negocio (Revenue)
revenue = pedidos_validos['monto_total'].sum()

print(f"Ingreso total (Revenue): ${revenue:,.2f}")


Ingreso total (Revenue): $51,965,834.26


In [15]:

# Unir pedidos con el catálogo para obtener el costo unitario
orders_cost = orders.merge(
    catalog[['nombre_producto', 'costo_unitario']],
    on='nombre_producto',
    how='left'
)

# Calcular el costo total de los productos vendidos
costo_total = (
    orders_cost['cantidad'] * orders_cost['costo_unitario']
).sum()

print(f"Costo total de los productos vendidos: ${costo_total:,.2f}")


Costo total de los productos vendidos: $43,124,018.41


In [16]:

# Inversión total en marketing
marketing_spend = marketing['gasto'].sum()

print(f"Inversión total en marketing: ${marketing_spend:,.2f}")


Inversión total en marketing: $2,871,843.53


In [17]:

# Calcular el profit del negocio
profit = revenue - costo_total - marketing_spend

print(f"Profit del negocio: ${profit:,.2f}")

# Calcular el margen de profit sobre los ingresos
margen_profit = (profit / revenue) * 100

print(f"Margen de profit: {margen_profit:.2f}%")


Profit del negocio: $5,969,972.32
Margen de profit: 11.49%


In [18]:

# Ticket promedio por orden
ticket_promedio = pedidos_validos['monto_total'].mean()

print(f"Ticket promedio por orden: ${ticket_promedio:,.2f}")


Ticket promedio por orden: $2,082.80


In [19]:

# Cantidad promedio de productos por orden
productos_por_orden = (
    pedidos_validos.groupby('id_pedido')['cantidad'].sum()
)

promedio_productos = productos_por_orden.mean()

print(
    f"Cantidad promedio de productos por orden: "
    f"{promedio_productos:.2f}"
)

Cantidad promedio de productos por orden: 7.12


In [20]:

# Producto más vendido por cantidad de unidades
ventas_por_producto = (
    orders.groupby('nombre_producto')['cantidad']
    .sum()
    .sort_values(ascending=False)
)

print("Unidades vendidas por producto:")
print(ventas_por_producto)

print(f"\nProducto más vendido: {ventas_por_producto.idxmax()}")
print(f"Unidades vendidas: {ventas_por_producto.max():,.0f}")


Unidades vendidas por producto:
nombre_producto
Laptop-Gaming-16GB      144198.0
Vacuum-Pro-Black          6284.0
Blender-XL-Red            6279.0
Jacket-Winter-M           6256.0
Sneakers-Urban-42         6172.0
Tablet-Standard-64GB      4153.0
Phone-Pro-128GB           4135.0
Name: cantidad, dtype: float64

Producto más vendido: Laptop-Gaming-16GB
Unidades vendidas: 144,198


In [21]:

# Gasto total en marketing por canal
gasto_por_canal = (
    marketing.assign(
        canal=marketing['canal'].fillna('Sin canal identificado')
    )
    .groupby('canal')['gasto']
    .sum()
    .sort_values(ascending=False)
)

print("Gasto en marketing por canal:")
print(gasto_por_canal.to_string())

print(f"\nGasto total en marketing: ${gasto_por_canal.sum():,.2f}")


Gasto en marketing por canal:
canal
social                    918043.21
organic                   913533.01
paid_search               863088.21
Sin canal identificado    177179.10

Gasto total en marketing: $2,871,843.53


In [22]:

# Revisar las cantidades más altas registradas
orders[
    ['id_pedido', 'nombre_producto', 'cantidad', 'precio_unitario', 'monto_total']
].sort_values(
    by='cantidad', ascending=False
).head(10)


,id_pedido,nombre_producto,cantidad,precio_unitario,monto_total
3668,order_3668,Laptop-Gaming-16GB,20000.0,348.31,6966200.0
3656,order_3656,Laptop-Gaming-16GB,20000.0,297.66,5953200.0
3726,order_3726,Laptop-Gaming-16GB,20000.0,290.85,5817000.0
3722,order_3722,Laptop-Gaming-16GB,20000.0,442.01,8840200.0
3521,order_3521,Laptop-Gaming-16GB,10000.0,43.14,431400.0
3748,order_3748,Laptop-Gaming-16GB,10000.0,336.93,3369300.0
3586,order_3586,Laptop-Gaming-16GB,10000.0,490.35,4903500.0
3643,order_3643,Laptop-Gaming-16GB,10000.0,238.15,2381500.0
3689,order_3689,Laptop-Gaming-16GB,10000.0,87.69,876900.0
3522,order_3522,Laptop-Gaming-16GB,10000.0,280.55,2805500.0


In [23]:

# Revisar los pedidos de Laptop-Gaming-16GB
orders.loc[
    orders['nombre_producto'] == 'Laptop-Gaming-16GB',
    ['id_pedido', 'nombre_producto', 'cantidad',
     'precio_unitario', 'monto_total']
].sort_values(
    by='cantidad', ascending=False
).head(10).to_string(index=False)


' id_pedido    nombre_producto  cantidad  precio_unitario  monto_total\norder_3726 Laptop-Gaming-16GB   20000.0           290.85    5817000.0\norder_3668 Laptop-Gaming-16GB   20000.0           348.31    6966200.0\norder_3656 Laptop-Gaming-16GB   20000.0           297.66    5953200.0\norder_3722 Laptop-Gaming-16GB   20000.0           442.01    8840200.0\norder_3748 Laptop-Gaming-16GB   10000.0           336.93    3369300.0\norder_3689 Laptop-Gaming-16GB   10000.0            87.69     876900.0\norder_3521 Laptop-Gaming-16GB   10000.0            43.14     431400.0\norder_3522 Laptop-Gaming-16GB   10000.0           280.55    2805500.0\norder_3586 Laptop-Gaming-16GB   10000.0           490.35    4903500.0\norder_3643 Laptop-Gaming-16GB   10000.0           238.15    2381500.0'

In [24]:

# Estadísticas de las cantidades vendidas
print("Estadísticas de cantidad:")
print(orders['cantidad'].describe())

print("\nCantidad máxima por producto:")
print(
    orders.groupby('nombre_producto')['cantidad']
    .max()
    .sort_values(ascending=False)
)


Estadísticas de cantidad:
count    24950.000000
mean         7.115110
std        296.869961
min         -2.000000
25%          1.000000
50%          2.000000
75%          2.000000
max      20000.000000
Name: cantidad, dtype: float64

Cantidad máxima por producto:
nombre_producto
Laptop-Gaming-16GB      20000.0
Blender-XL-Red              2.0
Jacket-Winter-M             2.0
Phone-Pro-128GB             2.0
Sneakers-Urban-42           2.0
Tablet-Standard-64GB        2.0
Vacuum-Pro-Black            2.0
Name: cantidad, dtype: float64


In [25]:

# Comprobar únicamente los registros con cantidades superiores a 2
cantidades_extremas = orders.loc[
    orders['cantidad'] > 2,
    ['id_pedido', 'nombre_producto', 'cantidad',
     'precio_unitario', 'monto_total']
]

print("Registros con cantidades superiores a 2:", len(cantidades_extremas))
print(cantidades_extremas.to_string(index=False))


Registros con cantidades superiores a 2: 10
 id_pedido    nombre_producto  cantidad  precio_unitario  monto_total
order_3521 Laptop-Gaming-16GB   10000.0            43.14     431400.0
order_3522 Laptop-Gaming-16GB   10000.0           280.55    2805500.0
order_3586 Laptop-Gaming-16GB   10000.0           490.35    4903500.0
order_3643 Laptop-Gaming-16GB   10000.0           238.15    2381500.0
order_3656 Laptop-Gaming-16GB   20000.0           297.66    5953200.0
order_3668 Laptop-Gaming-16GB   20000.0           348.31    6966200.0
order_3689 Laptop-Gaming-16GB   10000.0            87.69     876900.0
order_3722 Laptop-Gaming-16GB   20000.0           442.01    8840200.0
order_3726 Laptop-Gaming-16GB   20000.0           290.85    5817000.0
order_3748 Laptop-Gaming-16GB   10000.0           336.93    3369300.0


---

## 🔹 Paso 3: Entender dónde se pierden los usuarios (funnel de conversión)

**🎯 Objetivo:** Analizar el comportamiento de los usuarios para identificar en qué etapa del proceso se pierden.


⚙️**Conexión a la base de datos**:  
Se ejecuta la línea de configuración para conectar con la base de datos y aplicar consultas SQL en la tabla **events**.

---

**📊 Parte 1: Construcción del funnel**
- ¿Cuántos usuarios llegan a cada etapa del funnel?  
- Se calcula el número de usuarios únicos por `nombre_evento`  
- Se ordenan los eventos según el flujo del usuario  

---

**📉 Parte 2: Análisis de conversión**
- Se calcula la tasa de conversión entre cada paso del funnel  
- Se identifica en qué etapa se pierde la mayor cantidad de usuarios  
- ¿Cuál es la tasa de conversión final?
---

In [26]:
import pandas as pd
from sqlalchemy import create_engine

# ======================
# Conexión (NO modificar)
# ======================
db_config = {
    'user': 'practicum_student',
    'pwd': 'QnmDH8Sc2TQLvy2G3Vvh7',
    'host': 'yp-trainers-practicum.cluster-czs0gxyx2d8w.us-east-1.rds.amazonaws.com',
    'port': 5432,
    'db': 'data-analyst-production-db-en'
}

connection_string = 'postgresql://{}:{}@{}:{}/{}'.format(
    db_config['user'],
    db_config['pwd'],
    db_config['host'],
    db_config['port'],
    db_config['db']
)

engine = create_engine(connection_string, connect_args={'sslmode':'require'})

In [27]:
# Explorar tabla events
# =========================
query_events = '''
SELECT *
FROM events;
'''
events = pd.read_sql(query_events, con=engine)
events.head()

,id_usuario,id_sesion,nombre_evento,timestamp_evento,pais,dispositivo,fuente_referencia,categoria_producto
0,user_6772,6a97f2af-32ae-4186-8c92-04025be1a27b,first_visit,2025-05-17,Colombia,desktop,organic,Moda
1,user_5883,369b767c-1c33-4b2f-a652-c7c0ef92cfc9,add_to_cart,2025-02-23,Mexico,mobile,social,Hogar
2,user_5946,60039041-e78b-474c-87b3-c0b7e9c30708,add_payment_info,2025-05-15,Colombia,desktop,social,Electronica
3,user_827,18252a64-f389-4ef7-9e58-dadad4a3491e,purchase,2025-03-31,Mexico,mobile,social,Moda
4,user_2361,221b364e-cdc5-4668-b698-18d5ba849a67,first_visit,2025-01-22,Argentina,desktop,paid_search,Electronica


In [28]:

# PARTE 1: Totales del funnel
# ======================

query_totals = '''
SELECT
    nombre_evento,
    COUNT(DISTINCT id_usuario) AS usuarios_unicos
FROM events
GROUP BY nombre_evento
ORDER BY usuarios_unicos DESC;
'''

totals = pd.read_sql(query_totals, con=engine)
totals


,nombre_evento,usuarios_unicos
0,first_visit,7796
1,add_to_cart,7634
2,select_item,7582
3,begin_checkout,7208
4,add_payment_info,6250
5,purchase,6240


In [29]:

# PARTE 2: Conversiones
# ======================

query_conversion = '''
WITH funnel AS (
    SELECT
        nombre_evento,
        COUNT(DISTINCT id_usuario) AS usuarios_unicos
    FROM events
    WHERE nombre_evento IN (
        'first_visit',
        'add_to_cart',
        'select_item',
        'begin_checkout',
        'add_payment_info',
        'purchase'
    )
    GROUP BY nombre_evento
),
orden_funnel AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        CASE nombre_evento
            WHEN 'first_visit' THEN 1
            WHEN 'add_to_cart' THEN 2
            WHEN 'select_item' THEN 3
            WHEN 'begin_checkout' THEN 4
            WHEN 'add_payment_info' THEN 5
            WHEN 'purchase' THEN 6
        END AS orden
    FROM funnel
),
conversiones AS (
    SELECT
        nombre_evento,
        usuarios_unicos,
        LAG(usuarios_unicos) OVER (
            ORDER BY orden
        ) AS usuarios_etapa_anterior,
        orden
    FROM orden_funnel
)
SELECT
    nombre_evento,
    usuarios_unicos,
    usuarios_etapa_anterior,
    ROUND(
        100.0 * usuarios_unicos /
        NULLIF(usuarios_etapa_anterior, 0),
        2
    ) AS conversion_pct,
    usuarios_etapa_anterior - usuarios_unicos
        AS usuarios_perdidos,
    ROUND(
        100.0 * usuarios_unicos /
        NULLIF(
            FIRST_VALUE(usuarios_unicos) OVER (ORDER BY orden),
            0
        ),
        2
    ) AS conversion_acumulada_pct
FROM conversiones
ORDER BY orden;
'''

conversion = pd.read_sql(query_conversion, con=engine)
conversion


,nombre_evento,usuarios_unicos,usuarios_etapa_anterior,conversion_pct,usuarios_perdidos,conversion_acumulada_pct
0,first_visit,7796,NaN,NaN,NaN,100.00
1,add_to_cart,7634,7796.0,97.92,162.0,97.92
2,select_item,7582,7634.0,99.32,52.0,97.26
3,begin_checkout,7208,7582.0,95.07,374.0,92.46
4,add_payment_info,6250,7208.0,86.71,958.0,80.17
5,purchase,6240,6250.0,99.84,10.0,80.04


---

## 🔹 Paso 4: Evaluar si los usuarios regresan (retención por cohortes)

**🎯 Objetivo:** Analizar la retención de usuarios para entender si regresan después de registrarse.

**Tablas**

- `users` 
- `user_activity` 

---
1. Se identifica la cohorte de cada usuario según el **mes de registro**.


2. Se calcula la retención semanal: cuántos usuarios **se mantienen activos** en cada semana desde su registro.
   - `retenido_w1`: usuarios activos en la semana 1  
   - `retenido_w2`: usuarios activos en la semana 2  
   - `retenido_w3`: usuarios activos en la semana 3  


3. Se calcula el porcentaje de retención para cada semana, dividiendo los usuarios retenidos entre los clientes iniciales de la cohorte:  
   - `semana_1`: porcentaje de usuarios retenidos en la semana 1  
   - `semana_2`: porcentaje de usuarios retenidos en la semana 2  
   - `semana_3`: porcentaje de usuarios retenidos en la semana 3  

Se revisa que la columna de fecha esté en formato correcto (`DATE`).  
Se realiza la conversión usando: `CAST(fecha_registro AS DATE)`

In [30]:

# Explorar tabla users
# =========================
query_users = '''
SELECT *
FROM users;
'''

users = pd.read_sql(query_users, con=engine)
users.head(3)


,id_usuario,fecha_registro,país,dispositivo,tipo_plan
0,user_0,2025-01-29,Mexico,mobile,free
1,user_1,2025-01-07,Mexico,mobile,free
2,user_2,2025-03-12,Argentina,mobile,free


In [31]:

# Explorar tabla user_activity
# =========================
query_user_activity = '''
SELECT *
FROM user_activity;
'''

user_activity = pd.read_sql(query_user_activity, con=engine)
user_activity.head(3)


,id_usuario,fecha_actividad,dias_despues_registro,activo
0,user_0,2025-02-05,7,0
1,user_0,2025-02-12,14,1
2,user_0,2025-02-19,21,1


In [32]:

# Retención por cohortes
# ======================

query_cohort_retention_final = '''
WITH usuarios_cohorte AS (
    SELECT
        id_usuario,
        DATE_TRUNC('month', CAST(fecha_registro AS DATE)) AS cohorte
    FROM users
),
retencion AS (
    SELECT
        u.cohorte,
        COUNT(DISTINCT u.id_usuario) AS usuarios_iniciales,
        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 7 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w1,
        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 14 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w2,
        COUNT(DISTINCT CASE
            WHEN a.dias_despues_registro = 21 AND a.activo = 1
            THEN u.id_usuario
        END) AS retenido_w3
    FROM usuarios_cohorte AS u
    LEFT JOIN user_activity AS a
        ON u.id_usuario = a.id_usuario
    GROUP BY u.cohorte
)
SELECT
    cohorte,
    usuarios_iniciales,
    retenido_w1,
    retenido_w2,
    retenido_w3,
    ROUND(100.0 * retenido_w1 / NULLIF(usuarios_iniciales, 0), 2)
        AS semana_1,
    ROUND(100.0 * retenido_w2 / NULLIF(usuarios_iniciales, 0), 2)
        AS semana_2,
    ROUND(100.0 * retenido_w3 / NULLIF(usuarios_iniciales, 0), 2)
        AS semana_3
FROM retencion
ORDER BY cohorte;
'''

# Ejecutar la consulta
cohorte_final = pd.read_sql(query_cohort_retention_final, con=engine)
cohorte_final


,cohorte,usuarios_iniciales,retenido_w1,retenido_w2,retenido_w3,semana_1,semana_2,semana_3
0,2025-01-01 00:00:00+00:00,1627,697,668,656,42.84,41.06,40.32
1,2025-02-01 00:00:00+00:00,1444,611,609,635,42.31,42.17,43.98
2,2025-03-01 00:00:00+00:00,1636,677,705,690,41.38,43.09,42.18
3,2025-04-01 00:00:00+00:00,1606,680,697,663,42.34,43.40,41.28
4,2025-05-01 00:00:00+00:00,1687,695,676,706,41.20,40.07,41.85


---

## 🔹 Paso 5: Validar si los cambios generan impacto (test estadístico)

🎯 **Objetivo:** Evaluar si la modificación en la UI del checkout impacta la **tasa de conversión de compra**.

---

1. **Analizar el dataset** `experiment_checkout_ui.csv` para identificar la métrica principal **conversion**.
   - La métrica **conversion** es 1 si el usuario completó la compra, 0 si no.    
2. **Plantear la hipótesis estadística**     
3. **Aplicar el test estadístico adecuado** 
4. **Interpretar el resultado**  

### Hipótesis estadística

- **H₀ (hipótesis nula):** No existe una diferencia estadísticamente significativa entre las tasas de conversión del grupo de control y del grupo de tratamiento.
- **H₁ (hipótesis alternativa):** Existe una diferencia estadísticamente significativa entre las tasas de conversión del grupo de control y del grupo de tratamiento.

**Test estadístico:** Prueba Z para comparar dos proporciones independientes, con contraste bilateral.

**Nivel de significancia:** α = 0,05.

In [33]:
# tu código aquí
import pandas as pd
from io import BytesIO

url = "https://practicum-content.s3.amazonaws.com/datasets/experiment_checkout_ui.csv"

respuesta = requests.get(url, timeout=20)
respuesta.raise_for_status()

experiment_checkout_ui = pd.read_csv(BytesIO(respuesta.content))

print("Dimensiones:", experiment_checkout_ui.shape)
print("\nColumnas:")
print(experiment_checkout_ui.columns.tolist())

display(experiment_checkout_ui.head())

NameError: name 'requests' is not defined

In [ ]:
# Revisar información general del experimento
experiment_checkout_ui.info()

# Revisar valores ausentes
print("\nValores ausentes:")
print(experiment_checkout_ui.isnull().sum())

# Revisar registros duplicados
print("\nRegistros duplicados:", experiment_checkout_ui.duplicated().sum())

# Revisar los grupos del experimento
print("\nCantidad de usuarios por variante:")
print(experiment_checkout_ui['variante'].value_counts())

# Revisar los resultados de conversión
print("\nResultados de conversión:")
print(experiment_checkout_ui['convirtio'].value_counts())

In [34]:
# Calcular usuarios y conversiones por variante
conversion_por_variante = experiment_checkout_ui.groupby('variante').agg(
    usuarios=('id_usuario', 'count'),
    conversiones=('convirtio', 'sum'),
    tasa_conversion=('convirtio', 'mean')
)

conversion_por_variante['tasa_conversion'] *= 100

display(conversion_por_variante)

NameError: name 'experiment_checkout_ui' is not defined

In [35]:
from statsmodels.stats.proportion import proportions_ztest

# Conversiones y usuarios por variante
conversiones = conversion_por_variante['conversiones'].values
usuarios = conversion_por_variante['usuarios'].values

# Prueba Z para comparar dos proporciones
estadistico, p_valor = proportions_ztest(
    count=conversiones,
    nobs=usuarios,
    alternative='two-sided'
)

print(f"Estadístico Z: {estadistico:.4f}")
print(f"Valor p: {p_valor:.4f}")

NameError: name 'conversion_por_variante' is not defined

### Resultados de la prueba A/B

Se realizó una prueba Z para comparar las proporciones de conversión de los grupos de control y tratamiento, utilizando un nivel de significancia de 0,05.

La tasa de conversión del grupo de control fue del 15,69 %, mientras que la del grupo de tratamiento fue del 16,29 %, lo que representa una diferencia observada de aproximadamente 0,60 puntos porcentuales a favor del tratamiento.

La prueba estadística arrojó un estadístico Z de -0,8133 y un valor p de 0,4161. Como el valor p es superior al nivel de significancia establecido, no se rechaza la hipótesis nula.

Por lo tanto, no existe evidencia estadística suficiente para concluir que la nueva interfaz del checkout produzca un cambio significativo en la tasa de conversión. Con estos resultados, no se puede afirmar que la variante de tratamiento mejore la conversión de manera estadísticamente significativa.

In [36]:
print(orders.columns.tolist())
print(orders.head(2).to_string())

['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
  id_pedido id_usuario fecha_hora_pedido       pais dispositivo fuente_referencia       nombre_producto categoria_producto  cantidad  precio_unitario  monto_descuento  monto_total
0   order_0  user_6993        2025-05-22  Argentina     desktop           organic       Jacket-Winter-M               Moda       2.0           332.69              0.0       665.37
1   order_1  user_1329        2025-06-15     Mexico     desktop       paid_search  Tablet-Standard-64GB        Electrónica       1.0           176.86              5.0       171.86


In [37]:
# exportar datasets
orders.to_csv('orders_clean.csv', index=False, sep=',', encoding='utf-8')
catalog.to_csv('catalog_clean.csv', index=False, sep=',', encoding='utf-8')
marketing.to_csv('marketing_clean.csv', index=False, sep=',', encoding='utf-8')

---

In [38]:
with open('orders_clean.csv', 'r', encoding='utf-8') as archivo:
    for _ in range(3):
        print(archivo.readline().strip())

id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total
order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37
order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electrónica,1.0,176.86,5.0,171.86


In [39]:
# Crear copias compatibles con Tableau usando tabulaciones
orders.to_csv('orders_tableau.csv', index=False, sep='\t', encoding='utf-8')
catalog.to_csv('catalog_tableau.csv', index=False, sep='\t', encoding='utf-8')
marketing.to_csv('marketing_tableau.csv', index=False, sep='\t', encoding='utf-8')

In [40]:
import os

for archivo in [
    'orders_tableau.csv',
    'catalog_tableau.csv',
    'marketing_tableau.csv'
]:
    print(archivo, os.path.exists(archivo))

orders_tableau.csv True
catalog_tableau.csv True
marketing_tableau.csv True


In [41]:
import zipfile

with zipfile.ZipFile('archivos_tableau.zip', 'w') as zipf:
    zipf.write('orders_tableau.csv')
    zipf.write('catalog_tableau.csv')
    zipf.write('marketing_tableau.csv')

print("ZIP creado correctamente")

ZIP creado correctamente


In [42]:
print(orders_clean[['precio_unitario', 'monto_descuento', 'monto_total']].head())
print(orders_clean[['precio_unitario', 'monto_descuento', 'monto_total']].dtypes)

NameError: name 'orders_clean' is not defined

In [43]:
%whos

Variable                        Type         Data/Info
------------------------------------------------------
BytesIO                         type         <class '_io.BytesIO'>
archivo                         str          marketing_tableau.csv
cantidades_extremas             DataFrame           id_pedido     nomb<...>      336.93    3369300.0
catalog                         DataFrame            nombre_producto c<...>        Mcmillan-Rhodes  
cohorte_final                   DataFrame                        cohor<...>.20     40.07     41.85  
coinciden                       ndarray      24950: 24950 elems, type `bool`, 24950 bytes
columna                         str          categoria_producto
columnas_categoricas            list         n=4
columnas_nulas                  Index        Index(['pais', 'dispositi<...>],\n      dtype='object')
connection_string               str          postgresql://practicum_st<...>-analyst-production-db-en
conversion                      DataFrame       

In [44]:
import pandas as pd

orders_tableau = orders.copy()

columnas_numericas = [
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]

for columna in columnas_numericas:
    orders_tableau[columna] = pd.to_numeric(
        orders_tableau[columna], errors='coerce'
    )

print(orders_tableau[columnas_numericas].dtypes)
display(orders_tableau[columnas_numericas].head())

cantidad           float64
precio_unitario    float64
monto_descuento    float64
monto_total        float64
dtype: object


,cantidad,precio_unitario,monto_descuento,monto_total
0,2.0,332.69,0.0,665.37
1,1.0,176.86,5.0,171.86
2,2.0,102.99,10.0,195.99
3,1.0,257.87,15.0,242.87
4,1.0,336.28,0.0,336.28


In [45]:
orders_tableau.to_csv(
    'orders_tableau.csv',
    index=False,
    decimal='.',
    encoding='utf-8'
)

print("Archivo exportado correctamente.")

Archivo exportado correctamente.


In [46]:
verificacion = pd.read_csv('orders_tableau.csv')

display(verificacion[
    ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
].head())

print("\nTipos de datos:")
print(verificacion[
    ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
].dtypes)

print("\nValores nulos:")
print(verificacion[
    ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
].isna().sum())

,cantidad,precio_unitario,monto_descuento,monto_total
0,2.0,332.69,0.0,665.37
1,1.0,176.86,5.0,171.86
2,2.0,102.99,10.0,195.99
3,1.0,257.87,15.0,242.87
4,1.0,336.28,0.0,336.28



Tipos de datos:
cantidad           float64
precio_unitario    float64
monto_descuento    float64
monto_total        float64
dtype: object

Valores nulos:
cantidad           50
precio_unitario    50
monto_descuento    50
monto_total         0
dtype: int64


In [47]:
# Revisar los registros que tienen valores nulos
filas_nulas = verificacion[
    ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']
].isna().any(axis=1)

print("Filas con al menos un valor nulo:", filas_nulas.sum())

display(
    verificacion.loc[
        filas_nulas,
        [
            'id_pedido',
            'nombre_producto',
            'cantidad',
            'precio_unitario',
            'monto_descuento',
            'monto_total'
        ]
    ].head(10)
)

Filas con al menos un valor nulo: 50


,id_pedido,nombre_producto,cantidad,precio_unitario,monto_descuento,monto_total
74,order_74,Sneakers-Urban-42,NaN,NaN,NaN,595.85
75,order_75,Sneakers-Urban-42,NaN,NaN,NaN,458.15
76,order_76,Laptop-Gaming-16GB,NaN,NaN,NaN,319.75
77,order_77,Vacuum-Pro-Black,NaN,NaN,NaN,227.55
78,order_78,Phone-Pro-128GB,NaN,NaN,NaN,432.39
79,order_79,Sneakers-Urban-42,NaN,NaN,NaN,527.15
80,order_80,Vacuum-Pro-Black,NaN,NaN,NaN,711.18
81,order_81,Tablet-Standard-64GB,NaN,NaN,NaN,41.08
82,order_82,Sneakers-Urban-42,NaN,NaN,NaN,431.55
83,order_83,Sneakers-Urban-42,NaN,NaN,NaN,279.35


In [48]:

columnas = ['cantidad', 'precio_unitario', 'monto_descuento', 'monto_total']

print("Nulos en el DataFrame original:")
display(orders[columnas].isna().sum())

print("\nNulos en el archivo exportado:")
display(verificacion[columnas].isna().sum())


Nulos en el DataFrame original:


cantidad           50
precio_unitario    50
monto_descuento    50
monto_total         0
dtype: int64


Nulos en el archivo exportado:


cantidad           50
precio_unitario    50
monto_descuento    50
monto_total         0
dtype: int64

In [49]:

print("Dimensiones de los DataFrames disponibles:")

for nombre in ['orders', 'pedidos_validos', 'pedidos_completos']:
    if nombre in globals():
        df = globals()[nombre]
        print(f"{nombre}: {df.shape}")


Dimensiones de los DataFrames disponibles:
orders: (25000, 12)
pedidos_validos: (24950, 14)
pedidos_completos: (24950, 14)


In [50]:

print("Columnas de pedidos_validos:")
print(pedidos_validos.columns.tolist())

print("\nColumnas de pedidos_completos:")
print(pedidos_completos.columns.tolist())


Columnas de pedidos_validos:
['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'total_calculado', 'diferencia_total']

Columnas de pedidos_completos:
['id_pedido', 'id_usuario', 'fecha_hora_pedido', 'pais', 'dispositivo', 'fuente_referencia', 'nombre_producto', 'categoria_producto', 'cantidad', 'precio_unitario', 'monto_descuento', 'monto_total', 'total_calculado', 'diferencia_total']


In [51]:

for nombre, df in [
    ('pedidos_validos', pedidos_validos),
    ('pedidos_completos', pedidos_completos)
]:
    print(f"\n--- {nombre} ---")
    print("Filas:", len(df))
    print("Nulos en cantidad:", df['cantidad'].isna().sum())
    print("Nulos en precio_unitario:", df['precio_unitario'].isna().sum())
    print("Nulos en monto_descuento:", df['monto_descuento'].isna().sum())
    print("Nulos en total_calculado:", df['total_calculado'].isna().sum())
    print("Nulos en diferencia_total:", df['diferencia_total'].isna().sum())



--- pedidos_validos ---
Filas: 24950
Nulos en cantidad: 0
Nulos en precio_unitario: 0
Nulos en monto_descuento: 0
Nulos en total_calculado: 0
Nulos en diferencia_total: 0

--- pedidos_completos ---
Filas: 24950
Nulos en cantidad: 0
Nulos en precio_unitario: 0
Nulos en monto_descuento: 0
Nulos en total_calculado: 0
Nulos en diferencia_total: 0


In [52]:

print(
    "¿Son idénticos los dos DataFrames?",
    pedidos_validos.equals(pedidos_completos)
)

print("\nPrimeras filas de pedidos_validos:")
display(pedidos_validos.head(3))

print("\nPrimeras filas de pedidos_completos:")
display(pedidos_completos.head(3))


¿Son idénticos los dos DataFrames? False

Primeras filas de pedidos_validos:


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,total_calculado,diferencia_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,665.38,-0.01
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electrónica,1.0,176.86,5.0,171.86,171.86,0.00
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,195.98,0.01



Primeras filas de pedidos_completos:


,id_pedido,id_usuario,fecha_hora_pedido,pais,dispositivo,fuente_referencia,nombre_producto,categoria_producto,cantidad,precio_unitario,monto_descuento,monto_total,total_calculado,diferencia_total
0,order_0,user_6993,2025-05-22,Argentina,desktop,organic,Jacket-Winter-M,Moda,2.0,332.69,0.0,665.37,665.38,-0.01
1,order_1,user_1329,2025-06-15,Mexico,desktop,paid_search,Tablet-Standard-64GB,Electronica,1.0,176.86,5.0,171.86,171.86,0.00
2,order_2,user_3194,2025-05-02,Argentina,desktop,social,Blender-XL-Red,Hogar,2.0,102.99,10.0,195.99,195.98,0.01


In [53]:

print("Diferencias en cantidad de filas por categoría:")

print("\npedidos_validos:")
display(pedidos_validos['categoria_producto'].value_counts(dropna=False))

print("\npedidos_completos:")
display(pedidos_completos['categoria_producto'].value_counts(dropna=False))

print("\n¿Coinciden los identificadores y su orden?")
print(
    pedidos_validos['id_pedido'].reset_index(drop=True).equals(
        pedidos_completos['id_pedido'].reset_index(drop=True)
    )
)


Diferencias en cantidad de filas por categoría:

pedidos_validos:


Hogar          8346
Moda           8295
Electrónica    8279
NaN              30
Name: categoria_producto, dtype: int64


pedidos_completos:


Hogar          8346
Moda           8295
Electronica    8279
NaN              30
Name: categoria_producto, dtype: int64


¿Coinciden los identificadores y su orden?
True


In [54]:

for columna in pedidos_validos.columns:
    diferentes = (
        pedidos_validos[columna].fillna('__NULO__').astype(str)
        != pedidos_completos[columna].fillna('__NULO__').astype(str)
    ).sum()

    if diferentes > 0:
        print(f"{columna}: {diferentes} valores diferentes")


pais: 2479 valores diferentes
categoria_producto: 8279 valores diferentes


In [55]:

comparacion = pedidos_validos[
    ['id_pedido', 'pais', 'categoria_producto']
].merge(
    pedidos_completos[
        ['id_pedido', 'pais', 'categoria_producto']
    ],
    on='id_pedido',
    suffixes=('_validos', '_completos')
)

display(
    comparacion[
        (comparacion['pais_validos'] != comparacion['pais_completos']) |
        (
            comparacion['categoria_producto_validos'].fillna('__NULO__') !=
            comparacion['categoria_producto_completos'].fillna('__NULO__')
        )
    ].head(10)
)


,id_pedido,pais_validos,categoria_producto_validos,pais_completos,categoria_producto_completos
1,order_1,Mexico,Electrónica,Mexico,Electronica
3,order_3,Colombia,Electrónica,Colombia,Electronica
5,order_5,Mexico,Electrónica,mexico,Electronica
7,order_7,Mexico,Electrónica,Mexico,Electronica
8,order_8,Argentina,Electrónica,Argentina,Electronica
10,order_10,Mexico,Electrónica,Mexico,Electronica
11,order_11,Colombia,Electrónica,Colombia,Electronica
16,order_16,Colombia,Electrónica,Colombia,Electronica
18,order_18,Mexico,Electrónica,Mexico,Electronica
20,order_20,Argentina,Electrónica,Argentina,Electronica


In [56]:

print("DataFrame pedidos_validos:")
display(orders[['id_pedido', 'pais', 'categoria_producto']].head(10))

print("DataFrame pedidos_completos:")
display(pedidos_completos[['id_pedido', 'pais', 'categoria_producto']].head(10))


DataFrame pedidos_validos:


,id_pedido,pais,categoria_producto
0,order_0,Argentina,Moda
1,order_1,Mexico,Electrónica
2,order_2,Argentina,Hogar
3,order_3,Colombia,Electrónica
4,order_4,Argentina,Hogar
5,order_5,Mexico,Electrónica
6,order_6,Colombia,Hogar
7,order_7,Mexico,Electrónica
8,order_8,Argentina,Electrónica
9,order_9,Mexico,Moda


DataFrame pedidos_completos:


,id_pedido,pais,categoria_producto
0,order_0,Argentina,Moda
1,order_1,Mexico,Electronica
2,order_2,Argentina,Hogar
3,order_3,Colombia,Electronica
4,order_4,Argentina,Hogar
5,order_5,mexico,Electronica
6,order_6,Colombia,Hogar
7,order_7,Mexico,Electronica
8,order_8,Argentina,Electronica
9,order_9,Mexico,Moda


In [57]:

columnas_originales = orders.columns.tolist()

comparacion_original = orders.merge(
    pedidos_validos[['id_pedido']],
    on='id_pedido',
    how='inner'
)[columnas_originales].reset_index(drop=True)

comparacion_validos = pedidos_validos[
    columnas_originales
].reset_index(drop=True)

print(
    "¿pedidos_validos conserva los valores originales?",
    comparacion_original.equals(comparacion_validos)
)


¿pedidos_validos conserva los valores originales? True


In [58]:

# Preparar los datos para Tableau
orders_tableau = pedidos_validos.copy()

# Conservar las 12 columnas originales del proyecto
columnas_originales = orders.columns.tolist()
orders_tableau = orders_tableau[columnas_originales]

# Convertir las columnas numéricas a formato numérico
columnas_numericas = [
    'cantidad',
    'precio_unitario',
    'monto_descuento',
    'monto_total'
]

for columna in columnas_numericas:
    orders_tableau[columna] = pd.to_numeric(
        orders_tableau[columna],
        errors='coerce'
    )

# Exportar el archivo
orders_tableau.to_csv(
    'orders_tableau.csv',
    index=False,
    encoding='utf-8'
)

print("Filas exportadas:", len(orders_tableau))
print("Columnas exportadas:", len(orders_tableau.columns))
print("Valores nulos en columnas numéricas:")
display(orders_tableau[columnas_numericas].isna().sum())


Filas exportadas: 24950
Columnas exportadas: 12
Valores nulos en columnas numéricas:


cantidad           0
precio_unitario    0
monto_descuento    0
monto_total        0
dtype: int64

In [59]:

orders_tableau.to_csv(
    'orders_tableau.csv',
    index=False,
    sep=';',
    encoding='utf-8'
)

print("Archivo exportado con separador punto y coma.")


Archivo exportado con separador punto y coma.


In [60]:

with open('orders_tableau.csv', 'r', encoding='utf-8') as archivo:
    print(archivo.readline())


id_pedido;id_usuario;fecha_hora_pedido;pais;dispositivo;fuente_referencia;nombre_producto;categoria_producto;cantidad;precio_unitario;monto_descuento;monto_total



In [61]:

print(orders_tableau['monto_total'].head(10).tolist())
print(orders_tableau['monto_total'].dtype)


[665.37, 171.86, 195.99, 242.87, 336.28, 179.34, 327.19, 747.35, 949.54, 334.3]
float64


In [62]:

print("Ventas en orders:")
print(round(orders['monto_total'].sum(), 2))

print("\nVentas en pedidos_validos:")
print(round(pedidos_validos['monto_total'].sum(), 2))

print("\nVentas en orders_tableau:")
print(round(orders_tableau['monto_total'].sum(), 2))


Ventas en orders:
51987462.26

Ventas en pedidos_validos:
51965834.26

Ventas en orders_tableau:
51965834.26


In [63]:

print("Número de filas excluidas:", len(orders) - len(pedidos_validos))

print(
    "Ventas de las filas excluidas:",
    round(orders['monto_total'].sum() - pedidos_validos['monto_total'].sum(), 2)
)

print(
    "¿Los 50 registros excluidos tienen datos numéricos incompletos?",
    orders.loc[
        ~orders['id_pedido'].isin(pedidos_validos['id_pedido']),
        ['cantidad', 'precio_unitario', 'monto_descuento']
    ].isna().any(axis=1).all()
)


Número de filas excluidas: 50
Ventas de las filas excluidas: 21628.0
¿Los 50 registros excluidos tienen datos numéricos incompletos? True


## 🔹 Paso 6: Comunicar los resultados (Dashboard en BI)

🎯 **Objetivo**:  
Crear un dashboard que muestre de manera clara y visual los resultados del análisis de ventas, costos, marketing y conversión. 

Se usarán los CSVs limpios del Paso 1:

- `orders_clean.csv`  
- `catalog_clean.csv`  
- `marketing_clean.csv`

---

1️⃣ Preparación de los datos
1. Cargar los CSVs en Power BI o Tableau.
2. Revisar relaciones:
   - `orders.nombre_producto` → `catalog.nombre_producto`
   - `orders.fecha_pedido` → tabla de fechas (crear calendario para análisis temporal)
   - `orders.fecha_pedido` → `dim_fecha.date`
3. Crear columnas calculadas necesarias
4. Crear tabla de fechas para poder calcular comparaciones YTD, YoY o períodos anteriores (`Previous Year`, `Previous Month`).

---

2️⃣ Dashboard 1: Overview Ejecutivo
**KPIs principales a mostrar:**
- Revenue total
- Profit total
- Gasto total en marketing
- Ticket promedio
- Cantidad promedio de productos por orden

**Visualizaciones sugeridas:**
- Tarjetas KPI para revenue, profit y gasto marketing
- Gráfico de líneas: evolución mensual de revenue o profit
- Gráfico de líneas YTD
- Gráfico de barras: revenue y profit por producto o categoría

---

 3️⃣ Dashboard 2: Detalle / Drill-through  
**Objetivo:** Permitir explorar los datos desde el KPI general hasta cada orden o producto.

**Visualizaciones sugeridas:**
- Tabla detallada de órdenes con:
  - producto, cantidad, revenue, cost, profit
  - color condicional (profit negativo en rojo, positivo en verde)
- Gráfico de barras por producto con medida `cantidad vendida`
- Drill-through: seleccionar un producto y ver todos los pedidos relacionados
- Filtros por fecha, categoría de producto, etc

---

## Resumen ejecutivo

Este proyecto analiza el desempeño de RappiPlus a partir de información de pedidos, catálogo de productos, inversión en marketing y comportamiento de los usuarios. El objetivo es entender la rentabilidad del negocio, identificar oportunidades de mejora en el proceso de compra y evaluar la retención de clientes y el efecto de una modificación en la interfaz del checkout.

En el período analizado, los pedidos con información numérica completa registraron ingresos por **$51.965.834,26**. Al descontar los costos de producto y la inversión en marketing, la utilidad calculada fue de: **$5.969.972,32**, equivalente a un margen del **11,49 %**. La revisión de calidad permitió identificar duplicados, datos faltantes, diferencias puntuales en algunos importes y cantidades extremas que fueron investigadas y deben tenerse presentes al interpretar las ventas por producto.

El análisis del embudo muestra que la mayor pérdida de usuarios se presenta entre el inicio del checkout y el registro de la información de pago. Por su parte, el análisis por cohortes permite observar la actividad de los usuarios durante las primeras tres semanas posteriores al registro. Finalmente, la prueba estadística del experimento A/B no encontró evidencia suficiente para afirmar que la nueva interfaz del checkout mejore significativamente la conversión.

Los resultados ofrecen una base para orientar decisiones sobre rentabilidad, experiencia de compra y seguimiento de usuarios. Las conclusiones deben interpretarse teniendo en cuenta las limitaciones identificadas en los datos y los indicadores analizados.

## 🚀 Entrega Final

### 📎 **Dashboard Resumen Ejecutivo:** https://public.tableau.com/app/profile/catalina.duque/viz/ProyectoRappiplus_Final/ResumenEjecutivo?publish=yes&utm_source=chatgpt.com

### 📎 **Dashboard Detalle de Producto:** https://public.tableau.com/app/profile/catalina.duque/viz/ProyectoRappiplus_Final/ResumenEjecutivo?publish=yes&utm_source=chatgpt.com

